# Notebook 17: Operations Command Center (Phase 3B Capstone)

**Question:** What should an operations executive see first about CPAP resupply, and can every number on that screen be trusted?

**Sources (governed views only):** the shareable CPAP view (Notebook 14), the pipeline run log (Notebook 15), and AI summaries with their verification results (Notebook 16).

**Rule:** No number appears on the dashboard unless it was produced by a pipeline run that passed every quality gate.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.cc_item_summary_v
COMMENT 'Command center: CPAP resupply by item vs. policy allowance. From the shareable view only.'
AS
WITH items AS (
  SELECT HCPCS_CD,
         SUM(prescribers) AS prescribers,
         ROUND(SUM(median_per_patient * prescribers) / SUM(prescribers), 1) AS typical_per_patient,
         SUM(robust_high_flags) AS high_flags,
         SUM(robust_low_flags) AS low_flags
  FROM workspace.default.cpap_resupply_share_v
  GROUP BY HCPCS_CD),
limits AS (
  SELECT * FROM VALUES ('A4604',4),('A7030',4),('A7031',12),('A7032',24),('A7033',24),('A7034',4),
                       ('A7035',2),('A7036',2),('A7037',4),('A7038',24),('A7039',2),('A7046',2)
  AS t(HCPCS_CD, annual_max))
SELECT i.*, l.annual_max,
       ROUND(i.typical_per_patient / l.annual_max * 100, 0) AS pct_of_allowance
FROM items i JOIN limits l USING (HCPCS_CD)
""")
display(spark.table("workspace.default.cc_item_summary_v").orderBy("pct_of_allowance"))

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.cc_pipeline_health_v
COMMENT 'Command center: data freshness and the latest successful pipeline run.'
AS
WITH last_run AS (
  SELECT run_id, MAX(run_at) AS finished_at
  FROM workspace.default.pipeline_run_log
  WHERE gate = '9_run_complete' AND run_id NOT LIKE 'test-%'
  GROUP BY run_id
  ORDER BY finished_at DESC
  LIMIT 1)
SELECT 2024 AS data_year,
       DATE'2024-12-31' AS data_through,
       ROUND(MONTHS_BETWEEN(current_date(), DATE'2024-12-31'), 0) AS data_age_months,
       l.finished_at AS last_successful_run,
       DATEDIFF(current_date(), CAST(l.finished_at AS DATE)) AS days_since_last_run,
       SUM(CASE WHEN g.status = 'PASS' THEN 1 ELSE 0 END) AS gates_passed,
       COUNT(*) AS gates_logged
FROM last_run l
JOIN workspace.default.pipeline_run_log g ON g.run_id = l.run_id
GROUP BY l.run_id, l.finished_at
""")
display(spark.table("workspace.default.cc_pipeline_health_v"))

In [0]:
import re
from pyspark.sql import functions as F

rows = spark.table("workspace.default.cpap_llm_judged").toPandas()
def numbers_in(t): return {float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", t)}

checks = []
for _, r in rows.iterrows():
    allowed = {float(r.prescribers), float(r.typical_per_patient), float(r.annual_max),
               float(r.high_flags), float(r.low_flags)}
    bad = sorted(numbers_in(r.summary) - allowed)
    checks.append((r.HCPCS_CD, len(bad) == 0, str(bad)))
spark.createDataFrame(checks, "HCPCS_CD string, numbers_verified boolean, unsupported string") \
     .write.mode("overwrite").saveAsTable("workspace.default.ai_number_checks")

reviews = [("A7034", "REVISE", "Says patients 'require' units; data shows units received"),
           ("A7037", "REVISE", "Says patients 'require' units; frames flags as prescription rates"),
           ("A7038", "REVISE", "Widens scope to 'CPAP resupply items like' filters"),
           ("A7032", "REVISE", "Frames statistical flags as prescribing 'habits'"),
           ("A7035", "REVISE", "Frames statistical flags as prescription rates")]
spark.createDataFrame([(c, d, n, "Ali Al-Khafaji") for c, d, n in reviews],
                      "HCPCS_CD string, decision string, note string, reviewer string") \
     .withColumn("reviewed_at", F.current_timestamp()) \
     .write.mode("overwrite").saveAsTable("workspace.default.ai_human_review")

spark.sql("""
CREATE OR REPLACE VIEW workspace.default.cc_ai_summaries_v
COMMENT 'Command center: AI summaries shown only after human approval.'
AS
SELECT j.HCPCS_CD, j.item, n.numbers_verified, j.verdict AS ai_auditor,
       COALESCE(h.decision, 'PENDING') AS human_review, h.note AS review_note,
       CASE WHEN h.decision = 'APPROVED' THEN j.summary
            ELSE 'Withheld pending human review' END AS summary_for_display
FROM workspace.default.cpap_llm_judged j
JOIN workspace.default.ai_number_checks n USING (HCPCS_CD)
LEFT JOIN workspace.default.ai_human_review h USING (HCPCS_CD)
""")
display(spark.table("workspace.default.cc_ai_summaries_v").orderBy("HCPCS_CD"))

In [0]:
reviews = [
    ("A7030", "APPROVED", "Accurate description of flags"),
    ("A7036", "APPROVED", "Neutral wording: units per patient"),
    ("A4604", "REVISE", "'Usage' claims more than units received"),
    ("A7031", "REVISE", "'Usage' and 'prescribing patterns' overstate the data"),
    ("A7033", "REVISE", "'Usage' and 'significant margin' overstate the data"),
    ("A7039", "REVISE", "'Usage' and 'significantly' overstate the data"),
    ("A7046", "REVISE", "Says patients 'require' units; data shows units received"),
    ("A7034", "REVISE", "Says patients 'require' units; data shows units received"),
    ("A7037", "REVISE", "Says patients 'require' units; frames flags as prescription rates"),
    ("A7038", "REVISE", "Widens scope to 'CPAP resupply items like' filters"),
    ("A7032", "REVISE", "Frames statistical flags as prescribing 'habits'"),
    ("A7035", "REVISE", "Frames statistical flags as prescription rates")]
spark.createDataFrame([(c, d, n, "Ali Al-Khafaji") for c, d, n in reviews],
                      "HCPCS_CD string, decision string, note string, reviewer string") \
     .withColumn("reviewed_at", F.current_timestamp()) \
     .write.mode("overwrite").saveAsTable("workspace.default.ai_human_review")

spark.sql("""
CREATE OR REPLACE VIEW workspace.default.cc_item_statements_v
COMMENT 'Command center: factual statements built from a fixed template, not AI.'
AS
WITH names AS (SELECT * FROM VALUES
  ('A4604','Heated tubing'),('A7030','Full face mask'),('A7031','Full face mask cushion'),
  ('A7032','Nasal cushion'),('A7033','Nasal pillows'),('A7034','Nasal mask'),('A7035','Headgear'),
  ('A7036','Chinstrap'),('A7037','Standard tubing'),('A7038','Disposable filter'),
  ('A7039','Non-disposable filter'),('A7046','Humidifier water chamber') AS t(HCPCS_CD, item_name))
SELECT s.HCPCS_CD, n.item_name, s.prescribers, s.pct_of_allowance, s.high_flags, s.low_flags,
       CONCAT(n.item_name, ': patients received a typical ', FORMAT_NUMBER(s.typical_per_patient, 1),
              ' units per year, ', CAST(s.pct_of_allowance AS INT), '% of the ', s.annual_max,
              ' Medicare allows. Prescribers far above peers: ', s.high_flags, '; far below: ', s.low_flags, '.') AS statement
FROM workspace.default.cc_item_summary_v s JOIN names n USING (HCPCS_CD)
""")
display(spark.table("workspace.default.cc_item_statements_v").select("item_name", "statement").orderBy("pct_of_allowance"))
print("AI summaries approved for display:",
      spark.table("workspace.default.cc_ai_summaries_v").filter("human_review = 'APPROVED'").count())